In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os

spark = SparkSession.builder.getOrCreate()

BRONZE_PATH = "output/bronze"
SILVER_PATH = "output/silver"

In [2]:
df_orders = spark.read.csv(f"{BRONZE_PATH}/orders.csv", header=True)

orders = df_orders  \
    .dropDuplicates()  \
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp"))  \
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
    .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp")) \
    .withColumn("_is_valid", 
        col("order_purchase_timestamp").isNotNull()
    )

orders.show(n=5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|f4471dae8c482f51aa1826cd9f5d4433|167b9485947ed0a354a3f8dad04eb199|delivered   |2018-07-05 18:40:47     |2018-07-05 18:55:15|2018-07-10 15:10:00         |2018-07-11 21:16:47          |2018-07-1

In [3]:
window_orders = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())

orders = orders \
    .withColumn("rn", row_number().over(window_orders)) \
    .filter("rn = 1") \
    .drop("rn") \

orders.show(n=5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|00024acbcdf0a6daa1e931b038114c75|d4eb9395c8c0431ee92fce09860c5a06|delivered   |2018-08-08 10:00:35     |2018-08-08 10:10:18|2018-08-10 13:28:00         |2018-08-14 13:32:39          |2018-08-2

In [4]:
orders = orders.withColumn(
    "_is_valid",
    (col("order_id").isNotNull()) &
    (
        col("order_delivered_customer_date").isNull() |
        (col("order_delivered_customer_date") >= col("order_purchase_timestamp"))
    )
)

orders.show(n=5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+----------------+---------+
|00024acbcdf0a6daa1e931b038114c75|d4eb9395c8c0431ee92fce09860c5a06|delivered   |2018-08-08 10:00:35     |2018-08-08 10:10:18|2018-08-10 13:28:00         |2018-08-14 13:32:39          |2018-08-2

In [5]:
orders.write.mode("overwrite").csv(f"{SILVER_PATH}/orders", header=True)

In [6]:
df_order_items = spark.read.csv(f"{BRONZE_PATH}/order_items.csv", header=True)

order_items = df_order_items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

window_items = Window.partitionBy("order_id", "order_item_id") \
    .orderBy(col("_ingested_at").desc())

order_items = order_items \
    .withColumn("rn", row_number().over(window_items)) \
    .filter("rn = 1") \
    .drop("rn")

order_items.show(n=5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+-------------------------+----------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at             |_source_endpoint|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+-------------------------+----------------+
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa32f14722fc|9d7a1d34a5052409006425275ba1c2b4|2018-08-15 10:10:18|12.99|12.79        |2026-03-26 20:56:10.97396|order_items     |
|000576fe39319847cbb9d288c5617fa6|1            |557d850972a7d6f792fd18ae1400d9b6|5996cddab893a4652a15592fb58ab8db|2018-07-10 12:30:45|810.0|70.75        |2026-03-26 20:56:10.97396|order_items     |
|0005f5044

In [7]:
valid_orders = orders.select("order_id").distinct()

order_items = order_items.join(valid_orders, "order_id", "left")

order_items.show(n=5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+-------------------------+----------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at             |_source_endpoint|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+-------------------------+----------------+
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa32f14722fc|9d7a1d34a5052409006425275ba1c2b4|2018-08-15 10:10:18|12.99|12.79        |2026-03-26 20:56:10.97396|order_items     |
|000576fe39319847cbb9d288c5617fa6|1            |557d850972a7d6f792fd18ae1400d9b6|5996cddab893a4652a15592fb58ab8db|2018-07-10 12:30:45|810.0|70.75        |2026-03-26 20:56:10.97396|order_items     |
|0005f5044

In [8]:
order_items = order_items.withColumn(
    "_is_valid",
    (col("price") >= 0) &
    col("order_id").isNotNull()
)

In [9]:
order_items.write.mode("overwrite").csv(f"{SILVER_PATH}/order_items", header=True)

In [10]:
df_customers = spark.read.csv(f"{BRONZE_PATH}/customers.csv", header=True)
customers = df_customers \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

window_customers = Window.partitionBy("customer_id") \
    .orderBy(col("_ingested_at").desc())

customers = customers \
    .withColumn("rn", row_number().over(window_customers)) \
    .filter("rn = 1") \
    .drop("rn")

customers = customers.withColumn(
    "_is_valid",
    col("customer_id").isNotNull() &
    col("customer_unique_id").isNotNull()
)

customers.write.mode("overwrite").csv(f"{SILVER_PATH}/customers", header=True)

customers.show(n=5, truncate=False)

+--------------------------------+--------------------------------+------------------------+-------------------+--------------+--------------------------+----------------+---------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+--------------------------+----------------+---------+
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui               |RS            |2026-03-26 20:57:29.294769|customers       |true     |
|000598caf2ef4117407665ac33275130|7e0516b486e92ed3f3afdd6d1276cfbd|35540                   |oliveira           |MG            |2026-03-26 20:57:29.294769|customers       |true     |
|0013cd8e350a7cc76873441e431dd5ee|334fed5abcee3aa96c13f1432703e1fd|03585                  

In [11]:
df_products = spark.read.csv(f"{BRONZE_PATH}/products.csv", header=True)

products = df_products \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

window_products = Window.partitionBy("product_id") \
    .orderBy(col("_ingested_at").desc())

products = products \
    .withColumn("rn", row_number().over(window_products)) \
    .filter("rn = 1") \
    .drop("rn")

products = products.withColumn(
    "_is_valid",
    col("product_id").isNotNull()
)

products.write.mode("overwrite").csv(f"{SILVER_PATH}/products", header=True)

products.show(n=5, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------+----------------+---------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------+----------------+---------+
|00066f42aeeb9f3007548bb9d3f33c38|perfumaria           |53                 |596                       |6                 |300             |20               |16               |16              |2026-03-26 20:57:53.741227|products        |true     |
|00088930e92

In [12]:
df_sellers = spark.read.csv(f"{BRONZE_PATH}/sellers.csv", header=True)

sellers = df_sellers \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

window_sellers = Window.partitionBy("seller_id") \
    .orderBy(col("_ingested_at").desc())

sellers = sellers \
    .withColumn("rn", row_number().over(window_sellers)) \
    .filter("rn = 1") \
    .drop("rn")

sellers = sellers.withColumn(
    "_is_valid",
    col("seller_id").isNotNull()
)

sellers.write.mode("overwrite").csv(f"{SILVER_PATH}/sellers", header=True)

sellers.show(n=5, truncate=False)

+--------------------------------+----------------------+-----------+------------+--------------------------+----------------+---------+
|seller_id                       |seller_zip_code_prefix|seller_city|seller_state|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+----------------------+-----------+------------+--------------------------+----------------+---------+
|0015a82c2db000af6aaaf3ae2ecb0532|09080                 |santo andre|SP          |2026-03-26 20:57:53.926633|sellers         |true     |
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES          |2026-03-26 20:57:53.926633|sellers         |true     |
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ          |2026-03-26 20:57:53.926633|sellers         |true     |
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP          |2026-03-26 20:57:53.926633|sellers         |true     |
|003554e2dce176b5555353e4f3555ac8|74565  

In [13]:
df_payments = spark.read.csv(f"{BRONZE_PATH}/payments.csv", header=True)

payments = df_payments \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

window_payments = Window.partitionBy("order_id", "payment_sequential") \
    .orderBy(col("_ingested_at").desc())

payments = payments \
    .withColumn("rn", row_number().over(window_payments)) \
    .filter("rn = 1") \
    .drop("rn")

payments = payments.withColumn(
    "_is_valid",
    (col("payment_value") >= 0) &
    col("order_id").isNotNull()
)

payments.write.mode("overwrite").csv(f"{SILVER_PATH}/payments", header=True)

payments.show(n=5, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+----------------+---------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|_ingested_at              |_source_endpoint|_is_valid|
+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+----------------+---------+
|0009c9a17f916a706d71784483a5d643|1                 |credit_card |6                   |650.34       |2026-03-26 20:59:14.006964|payments        |true     |
|000e562887b1f2006d75e0be9558292e|1                 |credit_card |4                   |41.11        |2026-03-26 20:59:14.006964|payments        |true     |
|0010b2e5201cc5f1ae7e9c6cc8f5bd00|1                 |credit_card |3                   |65.5         |2026-03-26 20:59:14.006964|payments        |true     |
|0010dedd556712d7bb69a19cb7bbd37a|1                 |boleto     